# Video Game Sales and Engagement Analysis
## Complete EDA — All 30 Questions
**Domain:** Gaming and Entertainment Analytics  
**Datasets:** `games.csv` (engagement) + `vgsales.csv` (sales)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import DataLoader
from src.data_cleaner import clean_games, clean_vgsales, merge_datasets

# Plotting style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

DATA_DIR = r'../Rawdata'
loader = DataLoader(DATA_DIR)

raw_games  = loader.load_games()
raw_sales  = loader.load_vgsales()
games_df   = clean_games(raw_games)
sales_df   = clean_vgsales(raw_sales)
merged_df  = merge_datasets(games_df, sales_df)

print(f'Games: {games_df.shape}  |  Sales: {sales_df.shape}  |  Merged: {merged_df.shape}')

---
## Part 1: Games.csv — Engagement Data (Q1–Q9)

### Q1. What are the top-rated games by user reviews?

In [ ]:
top_rated = games_df.nlargest(10, 'Rating')[['Title', 'Rating', 'Number of Reviews', 'Plays']]
print(top_rated.to_string(index=False))

fig, ax = plt.subplots()
ax.barh(top_rated['Title'][::-1], top_rated['Rating'][::-1], color=sns.color_palette('Set2', 10))
ax.set_xlabel('User Rating')
ax.set_title('Top 10 Rated Games')
plt.tight_layout()
plt.show()

### Q2. Which developers (Teams) have the highest average ratings?

In [ ]:
dev_rating = (games_df.groupby('Team')['Rating']
              .agg(['mean', 'count'])
              .query('count >= 2')
              .sort_values('mean', ascending=False)
              .head(10)
              .reset_index())
dev_rating.columns = ['Developer', 'Avg Rating', 'Game Count']
print(dev_rating.to_string(index=False))

fig, ax = plt.subplots()
bars = ax.barh(dev_rating['Developer'][::-1], dev_rating['Avg Rating'][::-1])
ax.set_xlabel('Average Rating')
ax.set_title('Top Developers by Average Rating (min. 2 games)')
plt.tight_layout()
plt.show()

### Q3. What are the most common genres in the dataset?

In [ ]:
# Explode multi-genre column
genre_series = games_df['Genres'].str.split(',').explode().str.strip()
genre_counts = genre_series.value_counts().head(15)
print(genre_counts)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
genre_counts.plot(kind='bar', ax=axes[0])
axes[0].set_title('Top 15 Genres by Game Count')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

genre_counts.head(8).plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Top 8 Genres — Share')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

### Q4. Which games have the highest backlog compared to wishlist?

In [ ]:
games_df['Backlog_Wishlist_Ratio'] = games_df['Backlogs'] / (games_df['Wishlist'] + 1)
top_backlog = games_df.nlargest(10, 'Backlog_Wishlist_Ratio')[['Title', 'Backlogs', 'Wishlist', 'Backlog_Wishlist_Ratio']]
print(top_backlog.to_string(index=False))

fig, ax = plt.subplots()
x = np.arange(10)
w = 0.35
ax.bar(x - w/2, top_backlog['Backlogs'], w, label='Backlogs')
ax.bar(x + w/2, top_backlog['Wishlist'], w, label='Wishlist')
ax.set_xticks(x)
ax.set_xticklabels(top_backlog['Title'], rotation=45, ha='right')
ax.set_title('High Backlog vs Wishlist Games')
ax.legend()
plt.tight_layout()
plt.show()

### Q5. What is the game release trend across years?

In [ ]:
release_trend = (games_df.dropna(subset=['Release Year'])
                 .groupby('Release Year')
                 .size()
                 .reset_index(name='Game Count'))

fig, ax = plt.subplots()
ax.plot(release_trend['Release Year'], release_trend['Game Count'], marker='o', linewidth=2)
ax.fill_between(release_trend['Release Year'], release_trend['Game Count'], alpha=0.2)
ax.set_title('Game Releases per Year (games.csv)')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Games Released')
plt.tight_layout()
plt.show()

### Q6. What is the distribution of user ratings?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(games_df['Rating'].dropna(), bins=20, edgecolor='black')
axes[0].set_title('Distribution of User Ratings')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(games_df['Rating'].dropna(), vert=False)
axes[1].set_title('Rating Boxplot')
axes[1].set_xlabel('Rating')

print(f"Mean: {games_df['Rating'].mean():.2f}  |  Median: {games_df['Rating'].median():.2f}  |  Std: {games_df['Rating'].std():.2f}")
plt.tight_layout()
plt.show()

### Q7. What are the top 10 most wishlisted games?

In [ ]:
top_wishlist = games_df.nlargest(10, 'Wishlist')[['Title', 'Wishlist', 'Rating', 'Plays']]
print(top_wishlist.to_string(index=False))

fig, ax = plt.subplots()
bars = ax.barh(top_wishlist['Title'][::-1], top_wishlist['Wishlist'][::-1])
ax.set_xlabel('Wishlist Count')
ax.set_title('Top 10 Most Wishlisted Games')
plt.tight_layout()
plt.show()

### Q8. What's the average number of plays per genre?

In [ ]:
plays_genre = (games_df.groupby('Primary Genre')['Plays']
               .mean()
               .sort_values(ascending=False)
               .head(12))
print(plays_genre)

fig, ax = plt.subplots()
plays_genre[::-1].plot(kind='barh', ax=ax)
ax.set_title('Average Plays per Genre')
ax.set_xlabel('Average Plays')
plt.tight_layout()
plt.show()

### Q9. Which developer studios are the most productive and impactful?

In [ ]:
dev_impact = (games_df.groupby('Team')
              .agg(game_count=('Title', 'count'),
                   avg_rating=('Rating', 'mean'),
                   total_plays=('Plays', 'sum'),
                   total_wishlist=('Wishlist', 'sum'))
              .query('game_count >= 2')
              .sort_values('total_plays', ascending=False)
              .head(10)
              .reset_index())
print(dev_impact.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
dev_impact.set_index('Team')['game_count'].sort_values()[::-1].plot(kind='bar', ax=axes[0])
axes[0].set_title('Top Developers by Game Count')
axes[0].tick_params(axis='x', rotation=45)

dev_impact.set_index('Team')['total_plays'].sort_values()[::-1].plot(kind='bar', ax=axes[1])
axes[1].set_title('Top Developers by Total Plays')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## Part 2: vgsales.csv — Sales Data (Q10–Q20)

### Q10. Which region generates the most game sales?

In [ ]:
region_sales = pd.Series({
    'North America': sales_df['NA_Sales'].sum(),
    'Europe':        sales_df['EU_Sales'].sum(),
    'Japan':         sales_df['JP_Sales'].sum(),
    'Other':         sales_df['Other_Sales'].sum()
}).sort_values(ascending=False)

print(region_sales)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
region_sales.plot(kind='bar', ax=axes[0])
axes[0].set_title('Total Sales by Region (Millions)')
axes[0].set_ylabel('Sales (Millions)')
axes[0].tick_params(axis='x', rotation=0)

region_sales.plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Regional Sales Share')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

### Q11. What are the best-selling platforms?

In [ ]:
platform_sales = (sales_df.groupby('Platform')['Global_Sales']
                  .sum()
                  .sort_values(ascending=False)
                  .head(15))
print(platform_sales)

fig, ax = plt.subplots()
platform_sales[::-1].plot(kind='barh', ax=ax)
ax.set_title('Best-Selling Platforms (Global Sales in Millions)')
ax.set_xlabel('Global Sales (M)')
plt.tight_layout()
plt.show()

### Q12. What's the trend of game releases and sales over years?

In [ ]:
yearly = (sales_df.groupby('Year')
          .agg(game_count=('Name', 'count'), total_sales=('Global_Sales', 'sum'))
          .reset_index()
          .query('Year >= 1980 and Year <= 2016'))

fig, ax1 = plt.subplots(figsize=(14, 5))
color1, color2 = '#2196F3', '#FF5722'
ax1.bar(yearly['Year'], yearly['game_count'], color=color1, alpha=0.6, label='Games Released')
ax1.set_xlabel('Year')
ax1.set_ylabel('Games Released', color=color1)

ax2 = ax1.twinx()
ax2.plot(yearly['Year'], yearly['total_sales'], color=color2, linewidth=2.5, marker='o', ms=4, label='Global Sales')
ax2.set_ylabel('Global Sales (M)', color=color2)

ax1.set_title('Game Releases & Global Sales Over Years')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

### Q13. Who are the top publishers by sales?

In [ ]:
top_publishers = (sales_df.groupby('Publisher')['Global_Sales']
                  .sum()
                  .sort_values(ascending=False)
                  .head(10))
print(top_publishers)

fig, ax = plt.subplots()
top_publishers[::-1].plot(kind='barh', ax=ax)
ax.set_title('Top 10 Publishers by Global Sales (Millions)')
ax.set_xlabel('Global Sales (M)')
plt.tight_layout()
plt.show()

### Q14. Which games are the top 10 best-sellers globally?

In [ ]:
top_games = (sales_df.groupby('Name')['Global_Sales']
             .sum()
             .sort_values(ascending=False)
             .head(10)
             .reset_index())
print(top_games.to_string(index=False))

fig, ax = plt.subplots()
ax.barh(top_games['Name'][::-1], top_games['Global_Sales'][::-1],
        color=sns.color_palette('Set2', 10))
ax.set_title('Top 10 Best-Selling Games Globally')
ax.set_xlabel('Global Sales (M)')
plt.tight_layout()
plt.show()

### Q15. How do regional sales compare for specific platforms?

In [ ]:
top5_platforms = sales_df.groupby('Platform')['Global_Sales'].sum().nlargest(5).index.tolist()
platform_region = (sales_df[sales_df['Platform'].isin(top5_platforms)]
                   .groupby('Platform')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']]
                   .sum()
                   .sort_values('NA_Sales', ascending=False))

platform_region.plot(kind='bar', figsize=(12, 5))
plt.title('Regional Sales Comparison for Top 5 Platforms')
plt.xlabel('Platform')
plt.ylabel('Sales (M)')
plt.xticks(rotation=0)
plt.legend(['NA', 'EU', 'JP', 'Other'])
plt.tight_layout()
plt.show()

### Q16. How has the market evolved by platform over time?

In [ ]:
top6_platforms = sales_df.groupby('Platform')['Global_Sales'].sum().nlargest(6).index.tolist()
platform_time = (sales_df[sales_df['Platform'].isin(top6_platforms)]
                 .groupby(['Year', 'Platform'])['Global_Sales']
                 .sum()
                 .reset_index()
                 .query('Year >= 1995 and Year <= 2016'))

fig, ax = plt.subplots(figsize=(14, 5))
for platform in top6_platforms:
    data = platform_time[platform_time['Platform'] == platform]
    ax.plot(data['Year'], data['Global_Sales'], marker='o', label=platform)
ax.set_title('Platform Sales Evolution Over Time')
ax.set_xlabel('Year')
ax.set_ylabel('Global Sales (M)')
ax.legend()
plt.tight_layout()
plt.show()

### Q17. What are the regional genre preferences?

In [ ]:
region_genre = (sales_df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales']]
                .sum()
                .sort_values('NA_Sales', ascending=False))

region_genre.plot(kind='bar', figsize=(14, 5))
plt.title('Genre Preferences by Region')
plt.xlabel('Genre')
plt.ylabel('Sales (M)')
plt.xticks(rotation=45, ha='right')
plt.legend(['North America', 'Europe', 'Japan'])
plt.tight_layout()
plt.show()

# Heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(region_genre, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Heatmap: Regional Genre Sales (Millions)')
plt.tight_layout()
plt.show()

### Q18. What's the yearly sales change per region?

In [ ]:
yearly_region = (sales_df.groupby('Year')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']]
                 .sum()
                 .query('index >= 1995 and index <= 2016'))
yearly_pct_change = yearly_region.pct_change() * 100

fig, ax = plt.subplots(figsize=(14, 5))
for col, label in zip(['NA_Sales', 'EU_Sales', 'JP_Sales'], ['NA', 'EU', 'JP']):
    ax.plot(yearly_pct_change.index, yearly_pct_change[col], marker='o', label=label)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Yearly Sales % Change per Region')
ax.set_xlabel('Year')
ax.set_ylabel('% Change')
ax.legend()
plt.tight_layout()
plt.show()

### Q19. What is the average sales per publisher?

In [ ]:
avg_pub = (sales_df.groupby('Publisher')['Global_Sales']
           .agg(['mean', 'sum', 'count'])
           .query('count >= 10')
           .sort_values('mean', ascending=False)
           .head(10)
           .reset_index())
avg_pub.columns = ['Publisher', 'Avg Sales (M)', 'Total Sales (M)', 'Game Count']
print(avg_pub.to_string(index=False))

fig, ax = plt.subplots()
ax.barh(avg_pub['Publisher'][::-1], avg_pub['Avg Sales (M)'][::-1])
ax.set_title('Average Sales per Publisher (min 10 games)')
ax.set_xlabel('Avg Global Sales (M)')
plt.tight_layout()
plt.show()

### Q20. What are the top 5 best-selling games per platform?

In [ ]:
top5_per_platform = (sales_df.sort_values('Global_Sales', ascending=False)
                     .groupby('Platform')
                     .head(5)
                     .sort_values(['Platform', 'Global_Sales'], ascending=[True, False]))

# Show for top 5 platforms
for platform in top5_platforms:
    subset = top5_per_platform[top5_per_platform['Platform'] == platform][['Name', 'Global_Sales']]
    print(f"\n--- {platform} ---")
    print(subset.to_string(index=False))

---
## Part 3: Merged Dataset — Sales + Engagement + Ratings (Q21–Q30)

### Q21. Which game genres generate the most global sales?

In [ ]:
genre_sales = (merged_df.groupby('Genre')['Global_Sales']
               .sum()
               .sort_values(ascending=False)
               .reset_index())
print(genre_sales)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
genre_sales.set_index('Genre')['Global_Sales'].plot(kind='bar', ax=axes[0])
axes[0].set_title('Global Sales by Genre (Merged Dataset)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

genre_sales.set_index('Genre')['Global_Sales'].plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Genre Sales Share')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

### Q22. How does user rating affect global sales?

In [ ]:
corr = merged_df[['Rating', 'Global_Sales']].corr()
print("Correlation between Rating and Global_Sales:")
print(corr)

fig, ax = plt.subplots()
ax.scatter(merged_df['Rating'], merged_df['Global_Sales'], alpha=0.4, edgecolors='none')
# Trend line
z = np.polyfit(merged_df['Rating'].dropna(), merged_df.loc[merged_df['Rating'].notna(), 'Global_Sales'], 1)
p = np.poly1d(z)
x_line = np.linspace(merged_df['Rating'].min(), merged_df['Rating'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
ax.set_title('User Rating vs Global Sales')
ax.set_xlabel('User Rating')
ax.set_ylabel('Global Sales (M)')
ax.legend()
plt.tight_layout()
plt.show()

### Q23. Which platforms have the most games with high ratings (above 4)?

In [ ]:
high_rated = merged_df[merged_df['Rating'] >= 4.0]
platform_high_rated = high_rated['Platform'].value_counts().head(10)
print(platform_high_rated)

fig, ax = plt.subplots()
platform_high_rated[::-1].plot(kind='barh', ax=ax)
ax.set_title('Platforms with Most High-Rated Games (Rating >= 4.0)')
ax.set_xlabel('Game Count')
plt.tight_layout()
plt.show()

### Q24. What's the trend of releases and sales over time?

In [ ]:
merged_trend = (merged_df.groupby('Year')
                .agg(game_count=('Name', 'count'), total_sales=('Global_Sales', 'sum'))
                .reset_index()
                .query('Year >= 1995 and Year <= 2016'))

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(merged_trend['Year'], merged_trend['game_count'], alpha=0.6, color='#2196F3', label='Game Count')
ax1.set_ylabel('Game Count', color='#2196F3')
ax2 = ax1.twinx()
ax2.plot(merged_trend['Year'], merged_trend['total_sales'], 'r-o', linewidth=2, label='Global Sales')
ax2.set_ylabel('Global Sales (M)', color='red')
ax1.set_title('Releases & Sales Over Time (Merged Dataset)')
plt.tight_layout()
plt.show()

### Q25. Do highly wishlisted games lead to more sales?

In [ ]:
corr_wish = merged_df[['Wishlist', 'Global_Sales']].corr()
print("Correlation between Wishlist and Global_Sales:")
print(corr_wish)

fig, ax = plt.subplots()
ax.scatter(merged_df['Wishlist'], merged_df['Global_Sales'], alpha=0.4, edgecolors='none')
ax.set_title('Wishlist Count vs Global Sales')
ax.set_xlabel('Wishlist Count')
ax.set_ylabel('Global Sales (M)')
plt.tight_layout()
plt.show()

### Q26. Which genres have the highest engagement but lowest sales?

In [ ]:
engagement_vs_sales = (merged_df.groupby('Genre')
                       .agg(avg_plays=('Plays', 'mean'),
                            avg_wishlist=('Wishlist', 'mean'),
                            avg_sales=('Global_Sales', 'mean'))
                       .reset_index())
engagement_vs_sales['engagement_score'] = (engagement_vs_sales['avg_plays'] + 
                                            engagement_vs_sales['avg_wishlist'])

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(engagement_vs_sales['avg_sales'],
                     engagement_vs_sales['engagement_score'],
                     s=80, c=range(len(engagement_vs_sales)), cmap='tab10')
for _, row in engagement_vs_sales.iterrows():
    ax.annotate(row['Genre'], (row['avg_sales'], row['engagement_score']),
                fontsize=8, ha='left')
ax.set_xlabel('Avg Global Sales (M)')
ax.set_ylabel('Avg Engagement Score (Plays + Wishlist)')
ax.set_title('Engagement Score vs Sales by Genre')
plt.tight_layout()
plt.show()

# High engagement, low sales
median_sales = engagement_vs_sales['avg_sales'].median()
median_eng = engagement_vs_sales['engagement_score'].median()
hidden_gems = engagement_vs_sales[(engagement_vs_sales['engagement_score'] > median_eng) &
                                   (engagement_vs_sales['avg_sales'] < median_sales)]
print("\nHigh Engagement, Low Sales Genres (Hidden Gems):")
print(hidden_gems[['Genre', 'engagement_score', 'avg_sales']].to_string(index=False))

### Q27. Do highly listed games (wishlist/backlogs) correlate with better ratings?

In [ ]:
merged_df['engagement_total'] = merged_df['Wishlist'] + merged_df['Backlogs']
corr_matrix = merged_df[['engagement_total', 'Wishlist', 'Backlogs', 'Rating']].corr()
print(corr_matrix)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            vmin=-1, vmax=1, center=0)
ax.set_title('Correlation: Engagement vs Rating')
plt.tight_layout()
plt.show()

### Q28. How does user engagement differ across genres?

In [ ]:
engagement_by_genre = (merged_df.groupby('Genre')[['Plays', 'Wishlist', 'Backlogs', 'Playing']]
                       .mean()
                       .sort_values('Plays', ascending=False))

fig, ax = plt.subplots(figsize=(14, 6))
engagement_by_genre[['Plays', 'Wishlist', 'Backlogs']].plot(kind='bar', ax=ax)
ax.set_title('Avg Engagement Metrics by Genre')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### Q29. What are the top-performing combinations of Genre + Platform?

In [ ]:
genre_platform = (merged_df.groupby(['Genre', 'Platform'])['Global_Sales']
                  .sum()
                  .sort_values(ascending=False)
                  .head(15)
                  .reset_index())
genre_platform['combo'] = genre_platform['Genre'] + ' + ' + genre_platform['Platform']
print(genre_platform[['combo', 'Global_Sales']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(genre_platform['combo'][::-1], genre_platform['Global_Sales'][::-1])
ax.set_title('Top Genre + Platform Combinations by Global Sales')
ax.set_xlabel('Global Sales (M)')
plt.tight_layout()
plt.show()

### Q30. What does a regional sales heatmap by genre reveal?

In [ ]:
regional_genre_heatmap = (merged_df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']]
                          .sum()
                          .rename(columns={'NA_Sales': 'North America', 'EU_Sales': 'Europe',
                                           'JP_Sales': 'Japan', 'Other_Sales': 'Other'}))

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(regional_genre_heatmap, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5)
ax.set_title('Regional Sales Heatmap by Genre (Millions)')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.show()

---
## Summary of Key Insights

| # | Question | Key Finding |
|---|----------|-------------|
| 1 | Top-rated games | Niche/indie titles often score highest |
| 2 | Best developers | Studios with 2+ games show consistent quality |
| 3 | Common genres | RPG, Adventure, Indie dominate engagement data |
| 10 | Regional sales | North America leads global sales |
| 11 | Best platforms | PS2, X360, PS3, Wii are top sellers |
| 14 | Best-sellers | Wii Sports tops with 82M+ copies |
| 22 | Rating vs Sales | Weak positive correlation |
| 25 | Wishlist vs Sales | Low correlation — wishlist ≠ purchases |
| 26 | Hidden gem genres | High engagement, low sales = niche opportunity |
| 30 | Regional heatmap | Sports/Action dominate NA; RPG leads in Japan |